# Dice de YOLO11m-seg con instancias crudas, sin pintar (etapa 1, análisis de sensibilidad)
**Settings:** **CPU**, Internet OFF (no instala nada). Sin re-entrenar ni re-inferir.
**Inputs:**
- salida de `benchmark-segmentacion-cv-nino`: `results/<modelo>/fold<k>_preds.npz` (YOLO pintado con conf ≥ 0,25, como en el benchmark; SegFormer-B2 para la comparación) y `results/metricas_ic95.csv`;
- salida de `datasets-limpios-icdas-segmentacion`: `segmentation_manifest.csv` y `fdi_segmentation_curated.zip`;
- salida de `map-segmentadores`: `map/instancias.pkl.gz` (instancias crudas de YOLO en RLE, conf ≥ 0,001, resolución nativa).

Especificación: `claude/encargo_dice_yolo_crudo.md`. Celda de datos, labelmap y decodificador RLE copiados de `iou-caja-segmentadores` (`claude/nb_iou_caja_celdas.py`).

**Escenarios de YOLO** (por foto y clase FDI, la máscara predicha es):
- **pintado:** la del benchmark (instancias con conf ≥ 0,25 pintadas en orden de confianza creciente);
- **crudo (principal):** la instancia de mayor confianza de esa clase con conf ≥ 0,25 y ≥ 50 px, entera (misma regla que S2 del IoU de caja);
- **V1, unión:** unión de todas las instancias de esa clase con conf ≥ 0,25;
- **V2, sin umbral:** como crudo, con todas las instancias del caché (conf ≥ 0,001).

Las instancias crudas pueden superponerse: un píxel puede contar para dos dientes. **No es una máscara semántica**; es una sensibilidad de la regla de conversión y no reemplaza la comparación principal.

**Métricas = benchmark:** filas foto × clase con la clase en la referencia o predicha con ≥ 50 px; Dice e IoU por diente sumando intersección y áreas por clase sobre las fotos y promediando sobre los dientes; identificación (Dice ≥ 0,5 e IoU > 0,5): sensibilidad sobre dientes de referencia, VPP sobre clases predichas (≥ 50 px), F1; HD95 por diente (contornos, como en el benchmark) resumido por la mediana (la cifra del benchmark) y la media. Bootstrap de niños: 2.000 remuestreos, `default_rng(2026).integers(...)` (los mismos que `map-segmentadores` e `iou-caja-segmentadores`).

In [ ]:
import os, glob, gzip, pickle, time, zipfile, re
import numpy as np, pandas as pd, cv2
from scipy import ndimage
from concurrent.futures import ThreadPoolExecutor
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
MODELS = ['segformer_b2', 'yolo11m_seg']           # solo se leen estas predicciones
MIN_PX, CONF = 50, 0.25
B, SEED = 2000, 2026
TOL = 0.001
N_REF_ESPERADO, N_INST, N_INST025, N_FOUND_S2 = 9710, 14382, 10130, 9485
ESC = ['pintado', 'crudo', 'V1_union', 'V2_sin_umbral']
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
OUT = f'{W}/dice_yolo_crudo'; os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
T0 = time.time()

## Datos, labelmap y FDI (igual que en el benchmark)
Solo se leen las máscaras de referencia (no las fotos).

In [ ]:
def find(name):
    h = sorted(glob.glob(f'{IN}/**/{name}', recursive=True)); return h[0] if h else None
man = pd.read_csv(find('segmentation_manifest.csv'))
DATA = f'{TMP}/seg'
if not glob.glob(f'{DATA}/**/{man.mask_file[0]}', recursive=True):
    zipfile.ZipFile(find('fdi_segmentation_curated.zip')).extractall(DATA)
ROOT = os.path.dirname(os.path.dirname(glob.glob(f'{DATA}/**/{man.mask_file[0]}', recursive=True)[0]))
LM = (glob.glob(f'{DATA}/**/labelmap.txt', recursive=True) or [find('labelmap.txt')])[0]
lines = [l.strip() for l in open(LM, encoding='utf-8') if l.strip() and not l.startswith('#')]
NAMES, COLORS = [], []
for l in lines:
    p = l.split(':'); NAMES.append(p[0].strip()); COLORS.append(tuple(int(v) for v in p[1].split(',')) if len(p) > 1 and p[1] else None)
bg = [i for i, n in enumerate(NAMES) if n.lower() in ('background', 'fondo', 'bg')]
if bg and bg[0] != 0:
    i = bg[0]; NAMES.insert(0, NAMES.pop(i)); COLORS.insert(0, COLORS.pop(i))
NC = len(NAMES)
m0 = cv2.imread(os.path.join(ROOT, man.mask_file[0]), cv2.IMREAD_UNCHANGED); RGB_MASK = m0.ndim == 3
LUT = {c: i for i, c in enumerate(COLORS) if c is not None}
def read_mask(rel):
    m = cv2.imread(os.path.join(ROOT, rel), cv2.IMREAD_UNCHANGED)
    if not RGB_MASK: return m.astype(np.uint8)
    m = cv2.cvtColor(m[..., :3], cv2.COLOR_BGR2RGB); key = (m[..., 0].astype(np.int32) << 16) | (m[..., 1].astype(np.int32) << 8) | m[..., 2]
    out = np.zeros(m.shape[:2], np.uint8); seen = set(np.unique(key))
    for c, i in LUT.items():
        k = (c[0] << 16) | (c[1] << 8) | c[2]
        if k in seen: out[key == k] = i
    return out
with ThreadPoolExecutor(8) as ex:
    MASKS = dict(zip(man.image_id, ex.map(read_mask, man.mask_file)))
def univ(n):
    m = re.search(r'\d+', n); return int(m.group()) if m else None
def to_fdi(u):
    if u is None or not 1 <= u <= 32: return None
    if u <= 8: return 10 + (9 - u)
    if u <= 16: return 20 + (u - 8)
    if u <= 24: return 30 + (25 - u)
    return 40 + (u - 24)
U = [univ(n) for n in NAMES]; FDI = [to_fdi(u) if i else None for i, u in enumerate(U)]
assert NC == 33 and all(f is not None for f in FDI[1:])
TEETH = list(range(1, NC))
assert man.groupby('child_id').fold.nunique().max() == 1
def grupo(f):
    u = f % 10
    return {1: 'incisivos', 2: 'incisivos', 3: 'caninos', 4: 'premolares', 5: 'premolares', 6: 'molares 1.º–2.º', 7: 'molares 1.º–2.º', 8: 'terceros molares'}[u]
GRUPOS = ['incisivos', 'caninos', 'premolares', 'molares 1.º–2.º', 'terceros molares']
RES = os.path.dirname(find('metricas_ic95.csv')); REF = pd.read_csv(f'{RES}/metricas_ic95.csv').set_index('model')
for m in MODELS:
    for k in range(5): assert os.path.exists(f'{RES}/{m}/fold{k}_preds.npz'), f'falta {RES}/{m}/fold{k}_preds.npz'
print(len(MASKS), 'fotos |', man.child_id.nunique(), 'niños | clases', NC, '| raíz', ROOT, '| predicciones:', RES)
print(man.groupby('fold').agg(fotos=('image_id', 'size'), ninos=('child_id', 'nunique')).T.to_string())
print(REF[['dice_tooth', 'id_f1']].round(3).to_string())

## Caché de instancias crudas de YOLO y decodificador RLE
Decodificador del string comprimido de COCO copiado de `iou-caja-segmentadores` (allí verificado contra pycocotools); aquí se expande a máscara completa (orden por columnas). Si pycocotools está instalado, se compara con `mask.decode` en una muestra.

In [ ]:
def rle_counts(s):
    """Decodifica el string comprimido de COCO (igual que rleFrString de pycocotools)."""
    if isinstance(s, list): return s
    if isinstance(s, str): s = s.encode()
    cnts, p = [], 0
    while p < len(s):
        x, k, more = 0, 0, 1
        while more:
            c = s[p] - 48; x |= (c & 0x1f) << 5 * k; more = c & 0x20; p += 1; k += 1
            if not more and (c & 0x10): x |= -1 << 5 * k
        if len(cnts) > 2: x += cnts[-2]
        cnts.append(x)
    return cnts
def rle_decode(r):
    h, w = r['size']; cnts = np.asarray(rle_counts(r['counts']), np.int64)
    v = np.repeat(np.arange(len(cnts)) % 2 == 1, cnts)
    assert v.size == h * w, 'RLE con largo distinto de h*w'
    return v.reshape(w, h).T

INST = pickle.load(gzip.open(find('instancias.pkl.gz'))); Y = INST['yolo11m_seg']; del INST
assert len(Y) == len(man) and all(j in Y for j in man.image_id), 'el caché no tiene todas las fotos'
n_inst = sum(len(Y[j]['main']) for j in Y); n_inst025 = sum(s >= CONF for j in Y for _, s, _ in Y[j]['main'])
dec_msg = 'pycocotools no instalado: sin control cruzado'
try:
    from pycocotools import mask as MU
    smp = [q for j in list(Y)[:10] for _, _, q in Y[j]['main'][:4]]
    ok = all(np.array_equal(rle_decode(q), MU.decode(q).astype(bool)) for q in smp)
    dec_msg = f'decodificador vs pycocotools.decode en {len(smp)} máscaras: {"coincide" if ok else "NO COINCIDE"}'; assert ok
except ImportError: pass
print(f'instancias crudas de YOLO: {n_inst} (esperado {N_INST}); con conf ≥ {CONF}: {n_inst025} (esperado {N_INST025}) | {dec_msg}')

## Tabla por foto × clase para los 4 escenarios de YOLO y para SegFormer-B2
Para cada foto: máscaras por clase de cada escenario; filas = clases presentes en la referencia o predichas con ≥ 50 px (por escenario, como el benchmark). Se guardan intersección, píxeles de referencia y predichos, y HD95 por diente (percentil 95 de las distancias entre contornos, `binary_erosion` y `distance_transform_edt` como en el benchmark; se calcula en el recorte que contiene ambas máscaras más un borde de ceros, lo que da la misma distancia que en la imagen completa).
También se cuantifica la superposición de las instancias crudas con conf ≥ 0,25.

In [ ]:
def hd95(G, P):
    if not G.any() or not P.any(): return np.nan
    ys, xs = np.nonzero(G | P); s = (slice(ys.min(), ys.max() + 1), slice(xs.min(), xs.max() + 1))
    a, b = np.pad(G[s], 1), np.pad(P[s], 1)
    ea, eb = a ^ ndimage.binary_erosion(a), b ^ ndimage.binary_erosion(b)
    da, db = ndimage.distance_transform_edt(~eb), ndimage.distance_transform_edt(~ea)
    return np.percentile(np.concatenate([da[ea], db[eb]]), 95)

IDX = {f: c for c, f in enumerate(FDI) if f is not None}
def raw_masks(j, cmin, mode):
    """Máscaras por clase (índice) desde las instancias crudas: 'best' = mayor confianza con >= 50 px; 'union' = unión."""
    out = {}; by = {}
    for f, s, q in Y[j]['main']:
        if s >= cmin: by.setdefault(IDX[int(f)], []).append((s, q))
    for c, L in by.items():
        L.sort(key=lambda x: -x[0])
        if mode == 'union':
            m = np.zeros(MASKS[j].shape, bool)
            for _, q in L: m |= rle_decode(q)
            out[c] = m
        else:
            for _, q in L:
                m = rle_decode(q)
                if m.sum() >= MIN_PX: out[c] = m; break
    return out

rows, ovl = [], []
t = time.time()
for k in range(5):
    PY = np.load(f'{RES}/yolo11m_seg/fold{k}_preds.npz'); PS = np.load(f'{RES}/segformer_b2/fold{k}_preds.npz')
    for j in man[man.fold == k].image_id:
        g = MASKS[j]; py = PY[str(j)]; ps = PS[str(j)]
        assert py.shape == g.shape and ps.shape == g.shape, f'{j}: forma distinta'
        cy, cs, cg = (np.bincount(x.ravel(), minlength=NC)[:NC] for x in (py, ps, g))
        SC = {'pintado': {c: py == c for c in TEETH if cy[c] > 0}, 'SegFormer-B2': {c: ps == c for c in TEETH if cs[c] > 0},
              'crudo': raw_masks(j, CONF, 'best'), 'V1_union': raw_masks(j, CONF, 'union'), 'V2_sin_umbral': raw_masks(j, 0.0, 'best')}
        for esc, M in SC.items():
            for c in TEETH:
                P = M.get(c); npx = int(P.sum()) if P is not None else 0
                if cg[c] == 0 and npx < MIN_PX: continue
                G = g == c; inter = int((G & P).sum()) if P is not None else 0
                rows.append((esc, j, FDI[c], int(cg[c]), npx, inter, hd95(G, P) if (cg[c] and npx) else np.nan))
        # superposición entre instancias crudas con conf >= 0,25
        ms = [(int(f), rle_decode(q)) for f, s, q in Y[j]['main'] if s >= CONF]
        if ms:
            cov = np.sum([m for _, m in ms], axis=0)
            fd = pd.Series([f for f, _ in ms]).value_counts()
            ovl.append(dict(image_id=j, px_cubiertos=int((cov >= 1).sum()), px_en_2_o_mas=int((cov >= 2).sum()),
                            n_inst=len(ms), fdi_duplicados=int((fd > 1).sum())))
    del PY, PS
print(f'tabla: {time.time() - t:.0f} s')
T = pd.DataFrame(rows, columns=['escenario', 'image_id', 'fdi', 'gt_px', 'pred_px', 'inter', 'hd95'])
T = T.merge(man[['image_id', 'child_id', 'fold', 'view', 'condition']], on='image_id', how='left')
T['grupo'] = T.fdi.map(grupo)
T['in_gt'] = T.gt_px > 0; T['in_pred'] = T.pred_px >= MIN_PX
T['dice'] = 2 * T.inter / (T.gt_px + T.pred_px).clip(lower=1); T['iou'] = T.inter / (T.gt_px + T.pred_px - T.inter).clip(lower=1)
T['ok_dice'] = T.dice >= 0.5; T['ok_iou'] = T.iou > 0.5
OV = pd.DataFrame(ovl)
print(T.groupby('escenario').agg(filas=('fdi', 'size'), dientes_ref=('in_gt', 'sum'), clases_pred=('in_pred', 'sum')).to_string())

## Métricas (definiciones del benchmark) y chequeos
1. YOLO pintado y SegFormer-B2: Dice por diente y F1 (Dice ≥ 0,5) contra `metricas_ic95.csv` (±0,001). **Si no coinciden, el notebook se detiene.**
2. Instancias del caché (14.382; 10.130 con conf ≥ 0,25) y dientes de referencia encontrados con la regla cruda (9.485, como S2 del IoU de caja).
3. Informativo: F1 con IoU > 0,5 de YOLO pintado (0,945 en `f1-iou05`) y mediana de HD95 contra `metricas_ic95.csv`.

In [ ]:
def agg(d):
    g, p = d[d.in_gt], d[d.in_pred]
    pc = d.groupby('fdi').agg(I=('inter', 'sum'), S=('gt_px', 'sum'), Sp=('pred_px', 'sum'))
    pc['S'] = pc.S + pc.Sp; pc = pc[pc.S > 0]
    r = dict(dice_tooth=(2 * pc.I / pc.S).mean(), iou_tooth=(pc.I / (pc.S - pc.I)).mean(), n_ref=int(len(g)), n_pred=int(len(p)),
             n_found=int((g.pred_px >= MIN_PX).sum()))
    for crit in ('dice', 'iou'):
        se = g[f'ok_{crit}'].mean(); pp = p[f'ok_{crit}'].mean() if len(p) else np.nan
        r.update({f'sens_{crit}': se, f'ppv_{crit}': pp, f'f1_{crit}': 2 * se * pp / (se + pp) if se + pp > 0 else 0.0})
    h = d.hd95.dropna(); r.update(hd95_median=h.median(), hd95_mean=h.mean(), n_hd95=int(len(h)))
    return r
ALL = ESC + ['SegFormer-B2']
PT = {e: agg(T[T.escenario == e]) for e in ALL}
chk = pd.DataFrame([
    dict(chequeo='Dice por diente, YOLO pintado', valor=PT['pintado']['dice_tooth'], esperado=REF.loc['yolo11m_seg', 'dice_tooth'], tol=TOL),
    dict(chequeo='F1 (Dice >= 0,5), YOLO pintado', valor=PT['pintado']['f1_dice'], esperado=REF.loc['yolo11m_seg', 'id_f1'], tol=TOL),
    dict(chequeo='Dice por diente, SegFormer-B2', valor=PT['SegFormer-B2']['dice_tooth'], esperado=REF.loc['segformer_b2', 'dice_tooth'], tol=TOL),
    dict(chequeo='F1 (Dice >= 0,5), SegFormer-B2', valor=PT['SegFormer-B2']['f1_dice'], esperado=REF.loc['segformer_b2', 'id_f1'], tol=TOL),
    dict(chequeo='dientes de referencia', valor=PT['pintado']['n_ref'], esperado=N_REF_ESPERADO, tol=0),
    dict(chequeo='instancias crudas de YOLO en el caché', valor=n_inst, esperado=N_INST, tol=0),
    dict(chequeo='instancias con conf >= 0,25', valor=n_inst025, esperado=N_INST025, tol=0),
    dict(chequeo='dientes encontrados, crudo (= S2 IoU de caja)', valor=PT['crudo']['n_found'], esperado=N_FOUND_S2, tol=0),
    dict(chequeo='F1 (IoU > 0,5), YOLO pintado [informativo]', valor=PT['pintado']['f1_iou'], esperado=0.945, tol=0.0005),
    dict(chequeo='HD95 mediana, YOLO pintado [informativo]', valor=PT['pintado']['hd95_median'], esperado=REF.loc['yolo11m_seg', 'hd95'] if 'hd95' in REF else np.nan, tol=0.01)])
chk['ok'] = (chk.valor - chk.esperado).abs() <= chk.tol
chk.to_csv(f'{OUT}/dice_yolo_crudo_chequeos.csv', index=False); print(chk.to_string(index=False))
assert chk.ok.iloc[:4].all(), 'LAS PREDICCIONES NO REPRODUCEN metricas_ic95.csv: se detiene'
assert chk.ok.iloc[4], 'el número de dientes de referencia no es 9.710: se detiene'
print('chequeo principal OK' + ('' if chk.ok.iloc[5:8].all() else ' | OJO: algún conteo del caché no coincide (ver tabla)'))

## Tabla de YOLO (pintado, crudo, V1, V2) y SegFormer-B2: IC 95 % por bootstrap de niños y media ± DE por fold
Bootstrap con sumas por niño (equivale a concatenar los niños remuestreados): Dice e IoU por diente desde las sumas por niño y clase; F1 desde los conteos; HD95 como mediana ponderada por la multiplicidad del niño.

In [ ]:
KIDS = np.array(sorted(man.child_id.unique())); KI = {c: i for i, c in enumerate(KIDS)}
rng = np.random.default_rng(SEED)
DRAWS = rng.integers(0, len(KIDS), size=(B, len(KIDS)))      # idéntico a map-segmentadores / iou-caja-segmentadores
WK = np.stack([np.bincount(r, minlength=len(KIDS)) for r in DRAWS]).astype(float)
FD = sorted(T.fdi.unique()); FI = {f: i for i, f in enumerate(FD)}
def boot(d):
    kid = d.child_id.map(KI).to_numpy(); cl = d.fdi.map(FI).to_numpy()
    def m(v):
        a = np.zeros((len(KIDS), len(FD))); np.add.at(a, (kid, cl), v); return WK @ a
    I = m(d.inter.to_numpy(float)); S = m((d.gt_px + d.pred_px).to_numpy(float))
    with np.errstate(invalid='ignore', divide='ignore'):
        dice = np.nanmean(np.where(S > 0, 2 * I / S, np.nan), 1); iou = np.nanmean(np.where(S > 0, I / (S - I), np.nan), 1)
    def c(v): return WK @ np.bincount(kid, weights=v.astype(float), minlength=len(KIDS))
    r = dict(dice_tooth=dice, iou_tooth=iou)
    for crit in ('dice', 'iou'):
        se = c(d.in_gt & d[f'ok_{crit}']) / c(d.in_gt); pp = c(d.in_pred & d[f'ok_{crit}']) / c(d.in_pred)
        r[f'f1_{crit}'] = 2 * se * pp / (se + pp)
    h = d.dropna(subset=['hd95']).sort_values('hd95'); wv = WK[:, h.child_id.map(KI).to_numpy()]
    cw = np.cumsum(wv, 1); r['hd95_median'] = h.hd95.to_numpy()[np.argmax(cw >= cw[:, -1:] / 2, 1)]
    return pd.DataFrame(r)
MET = ['dice_tooth', 'iou_tooth', 'f1_dice', 'f1_iou', 'hd95_median']
BOOT, res, pf_rows = {}, [], []
for e in ALL:
    d = T[T.escenario == e]; bs = boot(d); BOOT[e] = bs
    pf = pd.DataFrame([dict(fold=k, **agg(x)) for k, x in d.groupby('fold')]); pf_rows += [dict(escenario=e, **r) for r in pf.to_dict('records')]
    row = dict(escenario=e, **PT[e])
    for m_ in MET:
        row[f'{m_}_lo'] = bs[m_].quantile(.025); row[f'{m_}_hi'] = bs[m_].quantile(.975)
        row[f'{m_}_fold_media'] = pf[m_].mean(); row[f'{m_}_fold_de'] = pf[m_].std()
    res.append(row)
res = pd.DataFrame(res); res.to_csv(f'{OUT}/dice_yolo_crudo_resumen.csv', index=False)
pd.DataFrame(pf_rows).to_csv(f'{OUT}/dice_yolo_crudo_por_fold.csv', index=False)
for _, r in res.iterrows():
    print(f'\n{r.escenario}  (dientes ref {r.n_ref}, encontrados {r.n_found}, clases predichas {r.n_pred})')
    for m_ in MET: print(f'   {m_:12s} {r[m_]:.3f} [{r[m_ + "_lo"]:.3f}–{r[m_ + "_hi"]:.3f}] | por fold {r[m_ + "_fold_media"]:.3f} ± {r[m_ + "_fold_de"]:.3f}')
    print(f'   hd95_mean    {r.hd95_mean:.2f} px')

## Comparaciones pareadas (bootstrap pareado de niños, mismos remuestreos)
- YOLO crudo − pintado, V1 − pintado y V2 − pintado (Dice por diente, IoU por diente, F1 con IoU > 0,5).
- SegFormer-B2 − YOLO crudo y SegFormer-B2 − YOLO pintado (mismas métricas): ¿se cierra la brecha de Dice (0,031)?
p bilateral = 2·min(P(Δ ≤ 0), P(Δ ≥ 0)); Holm dentro de cada métrica.

In [ ]:
PAIRS = [('crudo', 'pintado'), ('V1_union', 'pintado'), ('V2_sin_umbral', 'pintado'), ('SegFormer-B2', 'crudo'), ('SegFormer-B2', 'pintado')]
cmp = []
for m_ in ('dice_tooth', 'iou_tooth', 'f1_iou'):
    part = []
    for a, b in PAIRS:
        dl = BOOT[a][m_].to_numpy() - BOOT[b][m_].to_numpy()
        part.append(dict(metrica=m_, a=a, b=b, dif=PT[a][m_] - PT[b][m_], lo=np.quantile(dl, .025), hi=np.quantile(dl, .975),
                         p=min(1, 2 * min((dl <= 0).mean(), (dl >= 0).mean()))))
    part = pd.DataFrame(part).sort_values('p').reset_index(drop=True); n = len(part)
    part['p_holm'] = np.minimum(1, np.maximum.accumulate([(n - i) * p for i, p in enumerate(part.p)])); cmp.append(part)
cmp = pd.concat(cmp, ignore_index=True); cmp.to_csv(f'{OUT}/dice_yolo_crudo_pareada.csv', index=False)
print(cmp.round(4).to_string(index=False))
gap_p, gap_c = PT['SegFormer-B2']['dice_tooth'] - PT['pintado']['dice_tooth'], PT['SegFormer-B2']['dice_tooth'] - PT['crudo']['dice_tooth']
print(f'\nBrecha de Dice SegFormer − YOLO: pintado {gap_p:.3f} → crudo {gap_c:.3f} ({100 * (1 - gap_c / gap_p):.0f} % de la brecha explicada por pintar)')

## Dónde se gana o se pierde: grupo dentario, vista y retractor; dientes con cambio > 0,05

In [ ]:
sub = []
for col, lab in (('grupo', 'grupo dentario'), ('view', 'vista'), ('condition', 'retractor')):
    for (e, v), d in T.groupby(['escenario', col]):
        a = agg(d); sub.append(dict(variable=lab, nivel=v, escenario=e, n_fotos=d.image_id.nunique(), n_ninos=d.child_id.nunique(), **a))
sub = pd.DataFrame(sub); sub.to_csv(f'{OUT}/dice_yolo_crudo_subgrupos.csv', index=False)
for lab in ('grupo dentario', 'vista', 'retractor'):
    s = sub[sub.variable == lab]; tab = s.pivot_table(index='nivel', columns='escenario', values='dice_tooth')[ALL]
    tab['crudo − pintado'] = tab.crudo - tab.pintado; tab['n_ref'] = s.groupby('nivel').n_ref.first()
    print(f'\nDice por diente, {lab}:'); print(tab.round(3).to_string())
# por diente de referencia: crudo vs pintado
W_ = T[T.in_gt & T.escenario.isin(ESC)].pivot_table(index=['image_id', 'fdi'], columns='escenario', values='dice')
d_ = W_.crudo - W_.pintado
print(f'\nDientes de referencia: {len(d_)} | Dice crudo > pintado + 0,05: {(d_ > 0.05).sum()} ({(d_ > 0.05).mean():.1%}) | '
      f'crudo < pintado − 0,05: {(d_ < -0.05).sum()} ({(d_ < -0.05).mean():.1%}) | sin cambio (|Δ| ≤ 0,05): {(d_.abs() <= 0.05).mean():.1%}')

## Superposición de las instancias crudas (conf ≥ 0,25) y tabla por diente

In [ ]:
pct = OV.px_en_2_o_mas.sum() / OV.px_cubiertos.sum()
print(f'Píxeles cubiertos por instancias con conf ≥ 0,25 que caen en 2 o más instancias: {pct:.2%} '
      f'(mediana por foto {(OV.px_en_2_o_mas / OV.px_cubiertos).median():.2%})')
print(f'Pares foto–FDI con 2 o más instancias con conf ≥ 0,25: {OV.fdi_duplicados.sum()} en {(OV.fdi_duplicados > 0).sum()} de {len(OV)} fotos')
ovs = pd.DataFrame([dict(px_cubiertos=int(OV.px_cubiertos.sum()), px_en_2_o_mas=int(OV.px_en_2_o_mas.sum()), prop_px_superpuestos=pct,
                         pares_foto_fdi_duplicados=int(OV.fdi_duplicados.sum()), fotos_con_duplicados=int((OV.fdi_duplicados > 0).sum()), fotos=len(OV))])
chk2 = pd.concat([chk, pd.DataFrame([dict(chequeo=f'superposición: {k}', valor=v) for k, v in ovs.iloc[0].items()])], ignore_index=True)
chk2.to_csv(f'{OUT}/dice_yolo_crudo_chequeos.csv', index=False)
wide = T.pivot_table(index=['image_id', 'fdi'], columns='escenario', values=['pred_px', 'inter', 'dice', 'iou', 'hd95'])
wide.columns = [f'{v}_{e}' for v, e in wide.columns]
base = T.drop_duplicates(['image_id', 'fdi'])[['image_id', 'fdi', 'child_id', 'fold', 'view', 'condition', 'grupo', 'gt_px']].set_index(['image_id', 'fdi'])
base.join(wide).reset_index().to_csv(f'{OUT}/dice_yolo_crudo_por_diente.csv', index=False)
print('\ntiempo total', round((time.time() - T0) / 60, 1), 'min'); print(sorted(os.listdir(OUT)))